1: Importar librerías

In [8]:
# Capa Gold: resumen listo para Power BI.
# Silver conserva cada observación; Gold reúne métricas de lectura rápida.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 10, Finished, Available, Finished, False)

2: Comprobar la tabla histórica Silver

In [11]:
# 1. Leer el histórico Silver y comprobar su clave natural.
contexto = notebookutils.runtime.context
assert contexto.get("defaultLakehouseName"), (
    "Vincular un Lakehouse con esquemas como predeterminado."
)
historico = spark.table("silver.precios_combustibles_historico")
filas_silver = historico.count()
assert filas_silver > 0, "Silver está vacía: ejecuta antes la carga EIA."

duplicados_silver = historico.groupBy("serie_eia", "fecha").count() \
    .where(F.col("count") > 1).count()
assert duplicados_silver == 0, "Silver tiene series y fechas repetidas."

print(f"Silver: {filas_silver} observaciones válidas, {duplicados_silver} duplicados")

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 13, Finished, Available, Finished, False)

Silver: 3589 observaciones válidas, 0 duplicados


3: Crear las dimensiones de Combustible y Mes


In [12]:
# 2. Dimensiones: combustible y mes. Las claves son estables y únicas.
dim_combustible = historico.select(
    "serie_eia", "combustible", "region", "unidad"
).distinct()

assert dim_combustible.groupBy("serie_eia").count().where(F.col("count") > 1).count() == 0

base_mes = historico.select(
    F.trunc(F.col("fecha"), "MM").alias("fecha_mes")
).distinct()

dim_mes = base_mes.select(
    "fecha_mes",
    F.year("fecha_mes").alias("anio"),
    F.month("fecha_mes").alias("mes"),
    F.date_format("fecha_mes", "yyyy-MM").alias("etiqueta_mes"),
    F.quarter("fecha_mes").alias("trimestre"),
)

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 14, Finished, Available, Finished, False)

4: Crear la Tabla de Hechos Mensual (Agregación)

In [13]:
# 3. Hecho mensual: promedio por combustible y mes. El promedio no sustituye
# las observaciones semanales originales, que siguen en Silver.
mensual = historico.withColumn("fecha_mes", F.trunc(F.col("fecha"), "MM")) \
    .groupBy("serie_eia", "fecha_mes") \
    .agg(
        F.round(F.avg("precio_usd_galon"), 3).alias("precio_promedio_usd_galon"),
        F.min("precio_usd_galon").alias("precio_minimo_usd_galon"),
        F.max("precio_usd_galon").alias("precio_maximo_usd_galon"),
        F.count("*").cast("int").alias("semanas_observadas"),
    )

ventana_mes = Window.partitionBy("serie_eia").orderBy("fecha_mes")
mensual = mensual.withColumn(
    "precio_mes_anterior_usd_galon",
    F.lag("precio_promedio_usd_galon").over(ventana_mes),
).withColumn(
    "variacion_mensual_pct",
    F.when(F.col("precio_mes_anterior_usd_galon") > 0,
        F.round(100 * (F.col("precio_promedio_usd_galon") / 
                       F.col("precio_mes_anterior_usd_galon") - 1), 2)),
).select(
    "serie_eia", "fecha_mes", "precio_promedio_usd_galon",
    "precio_minimo_usd_galon", "precio_maximo_usd_galon",
    "semanas_observadas", "precio_mes_anterior_usd_galon",
    "variacion_mensual_pct",
)

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 15, Finished, Available, Finished, False)

5: Calcular el Último Precio Semanal (para tarjetas KPI)


In [14]:
# 4. Hecho de último precio semanal para las tarjetas KPI del informe.
ventana_ultima = Window.partitionBy("serie_eia").orderBy(F.col("fecha").desc())

ultimo = historico.withColumn("posicion", F.row_number().over(ventana_ultima)) \
    .withColumn("precio_anterior_usd_galon", F.lead("precio_usd_galon").over(ventana_ultima)) \
    .where(F.col("posicion") == 1) \
    .withColumn(
        "variacion_semanal_pct",
        F.when(F.col("precio_anterior_usd_galon") > 0,
            F.round(100 * (F.col("precio_usd_galon") / 
                           F.col("precio_anterior_usd_galon") - 1), 2)),
    ).select(
        "serie_eia", "fecha", "precio_usd_galon",
        "precio_anterior_usd_galon", "variacion_semanal_pct",
    )

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 16, Finished, Available, Finished, False)

6: Guardar las Tablas Delta en Gold


In [15]:
# Crear el esquema si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Guardar las 4 tablas anteponiendo 'gold.'
dim_combustible.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_combustible")
dim_mes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_mes")
mensual.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_precios_mensual")
ultimo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_ultimo_precio_semanal")

print("¡Capa Gold materializada exitosamente en Delta Lake con las 4 tablas!")

StatementMeta(, c845fea9-f0bb-42ef-9c89-51ef1025c52b, 17, Finished, Available, Finished, False)

¡Capa Gold materializada exitosamente en Delta Lake con las 4 tablas!
